# discriminator-classifier-head — faded example 1: Complete the flatten step of a scalar discriminator head

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `discriminator-classifier-head`. Running the beacon reports progress on the `GAN: Discriminator classifier head` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: Discriminator classifier head` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`discriminator-classifier-head`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "discriminator-classifier-head"
DD_SUBTOPIC = "GAN: Discriminator classifier head"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The DCGAN discriminator head flattens a `(B, C, H, W)` feature map to `(B, C*H*W)`, applies a bias-free `Linear(C*H*W, 1)`, then `Sigmoid`, then squeezes to `(B,)`. The flatten must collapse only the channel and spatial axes while preserving the batch axis and the left-to-right ordering the weight expects.

## Faded exercise 1

### Faded — flatten before the Linear

Implement `disc_head(features, weight)` which maps `features: (B, C, H, W)` to a `(B,)` real/fake probability vector using a bias-free linear head.

The Linear projection, sigmoid, and squeeze are already written. **You must complete only the flatten step** that turns `(B, C, H, W)` into `(B, C*H*W)`.

**Fill in:** Flatten the channel and spatial axes of `features` into one axis, keeping the batch axis, giving shape `(B, C*H*W)`.

In [ ]:
def disc_head(features: Tensor, weight: Tensor) -> Tensor:
    flat = rearrange(features, 'b c h w -> b (c h w)')   # (B, C*H*W)
    logits = flat @ weight.T               # (B, 1)
    return t.sigmoid(logits).squeeze(-1)   # (B,)


def _test():
    t.manual_seed(0)
    B, C, H, W = 4, 8, 5, 5
    features = t.randn(B, C, H, W)
    weight = t.randn(1, C * H * W)
    out = disc_head(features, weight)
    assert out.shape == (B,), f'expected ({B},), got {tuple(out.shape)}'
    assert bool(((out > 0) & (out < 1)).all()), 'probs must lie in (0,1)'
    flat_ref = features.flatten(start_dim=1)
    expected = t.sigmoid(flat_ref @ weight.T).squeeze(-1)
    assert t.allclose(out, expected, atol=1e-5), 'output disagrees with reference flatten+linear head'


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def disc_head(features: Tensor, weight: Tensor) -> Tensor:
    flat = rearrange(features, 'b c h w -> b (c h w)')   # (B, C*H*W)
    logits = flat @ weight.T               # (B, 1)
    return t.sigmoid(logits).squeeze(-1)   # (B,)
```
</details>